## Prepare ADRD dataset

In [1]:
## Load packages ----
import numpy as np
import pandas as pd
import sshtunnel
import psycopg2 as pg
import os

import json
import sys

import seaborn as sns
import matplotlib.pyplot as plt

/n/home_fasse/maudirac/.conda/envs/medicare_QC/lib/python3.9/site-packages/paramiko/transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


In [2]:
## read zip to county crosswalk ----
zip_to_county = pd.read_csv('../data/input/remote/zip_county_2010.csv')
zip_to_county = zip_to_county[['ZIP', 'COUNTY']]
zip_to_county = zip_to_county.rename(columns = {'ZIP':'zip', 'COUNTY':'county'})
zip_w = zip_to_county.groupby(['zip'])['county'].count().reset_index()
zip_w = zip_w.rename(columns = {'county':'w'})
zip_w['w'] = 1 / zip_w.w
zip_to_county = zip_to_county.merge(zip_w)
zip_to_county.w.describe()

count    46875.000000
mean         0.775168
std          0.278345
min          0.166667
25%          0.500000
50%          1.000000
75%          1.000000
max          1.000000
Name: w, dtype: float64

In [3]:
## Open ssh tunnel to DB host ----
tunnel = sshtunnel.SSHTunnelForwarder(
    ('nsaph.rc.fas.harvard.edu', 22),
    ssh_username=f'{os.environ["MY_NSAPH_SSH_USERNAME"]}',
    ssh_private_key=f'{os.environ["HOME"]}/.ssh/id_rsa', 
    ssh_password=f'{os.environ["MY_NSAPH_SSH_PASSWORD"]}', 
    remote_bind_address=("localhost", 5432)
)

tunnel.start()

## Open connection to DB ----
connection = pg.connect(
    host='localhost',
    database='nsaph2',
    user=f'{os.environ["MY_NSAPH_DB_USERNAME"]}',
    password=f'{os.environ["MY_NSAPH_DB_PASSWORD"]}', 
    port=tunnel.local_bind_port
)

In [4]:
## define functions ----
def get_outcomes(read_path):
    """ Get and return ICD codes """""
    f = open(read_path)
    res_dict = json.load(f)
    f.close()
    res_dict = json.loads(res_dict[0])
    return res_dict

def get_outcomes_set(outcome=None, year=None):
    """ Uses ICD9 for years prior 2015 and ICD10 otherwise """
    if year < 2015:
        outcomes_set = outcomes[outcome]["icd9"]
    elif year > 2015:
        outcomes_set = outcomes[outcome]["icd10"]
    else:
        outcomes_set = outcomes[outcome]["icd10"] + \
                       outcomes[outcome]["icd9"]
    return set(outcomes_set)

def get_outcome_in_diagnoses(outcomes_set=None, diagnoses=None):
    return any(o_ in outcomes_set for o_ in diagnoses)

## Beneficiary counts

In [5]:
## year range of interest ----
years_ = [y_ for y_ in range(2000, 2019)]
years_.remove(2015) # not available in DB as of Nov 2022
years_.remove(2006) # not available in DB as of Nov 2022

## obtain beneficiary counts per zipcode ----

bene_zip_list = list()

for y_ in years_: 
    print(y_)
    
    ## Define query ----
    sql_query = f"""
    SELECT 
        zip,
        year,
        race, 
        sex,
        case 
            when age < 65 then '<65'
            when age >= 65 and age < 75 then '[65,75)'
            when age >= 75 and age < 85 then '[75,85)'
            when age >= 85 then '>85'
        end age_grp,
        count(*) as n_enrollees
    FROM 
        medicare.enrollments
        LEFT JOIN medicare.beneficiaries ON medicare.enrollments.bene_id = medicare.beneficiaries.bene_id
    WHERE 
        year in ('{y_}') AND
        state = 'NC' AND 
        race in ('1', '2') AND
        sex in ('1', '2')
    GROUP BY 
        age_grp, 
        year, 
        zip, 
        race, 
        sex
    ;
    """
    ## Request query ----
    %time b = pd.read_sql_query(sql_query, connection, index_col = 'zip').reset_index()
    bene_zip_list.append(b)

2000
CPU times: user 350 ms, sys: 94.7 ms, total: 444 ms
Wall time: 1min 44s
2001
CPU times: user 108 ms, sys: 20.3 ms, total: 128 ms
Wall time: 18.2 s
2002
CPU times: user 104 ms, sys: 24.9 ms, total: 129 ms
Wall time: 17.9 s
2003
CPU times: user 93 ms, sys: 21.4 ms, total: 114 ms
Wall time: 18.7 s
2004
CPU times: user 99.2 ms, sys: 16.2 ms, total: 115 ms
Wall time: 19.3 s
2005
CPU times: user 94.7 ms, sys: 20.6 ms, total: 115 ms
Wall time: 19.8 s
2007
CPU times: user 116 ms, sys: 30.5 ms, total: 146 ms
Wall time: 1min 32s
2008
CPU times: user 91.1 ms, sys: 22.8 ms, total: 114 ms
Wall time: 19.6 s
2009
CPU times: user 93.6 ms, sys: 21 ms, total: 115 ms
Wall time: 17.5 s
2010
CPU times: user 96.7 ms, sys: 19.4 ms, total: 116 ms
Wall time: 17.8 s
2011
CPU times: user 95.6 ms, sys: 19.7 ms, total: 115 ms
Wall time: 19.8 s
2012
CPU times: user 98.8 ms, sys: 28.9 ms, total: 128 ms
Wall time: 47 s
2013
CPU times: user 92.6 ms, sys: 24.2 ms, total: 117 ms
Wall time: 20.5 s
2014
CPU times: us

In [6]:
## crosswalk to counties ----
bene_zip_df = pd.concat(bene_zip_list)
bene_county_df = bene_zip_df.merge(zip_to_county)
bene_county_df['n_enrollees'] = bene_county_df.n_enrollees * bene_county_df.w
bene_county_df = bene_county_df.groupby(['year', 'county', 'race', 'sex', 'age_grp'])['n_enrollees'].sum().reset_index()

In [7]:
## total number of enrollees in zipcodes ----
bene_zip_df.n_enrollees.sum()

25491220

In [8]:
## total number of enrollees in counties ----
bene_county_df.n_enrollees.sum()

24320746.0

## Admission counts

In [9]:
## year range of interest ----
years_ = [y_ for y_ in range(2000, 2019)]
years_.remove(2015) # not available in DB as of Nov 2022
years_.remove(2006) # not available in DB as of Nov 2022

## obtain adrd counts per zipcode ----
adm_zip_list = list()

for y_ in years_: 
    print(y_)
    
    ## Define query ----
    sql_query = f"""
    SELECT
        bene.bene_id,
        diagnoses,
        zip,
        year,
        race,
        sex, 
        EXTRACT(YEAR FROM dob) as yob_
    FROM 
        medicare.beneficiaries as bene
    RIGHT JOIN (
        SELECT 
            bene_id, 
            diagnoses, 
            zip, 
            year
        FROM 
            medicare.admissions as adm
        WHERE
            year in ('{y_}') AND
            state = 'NC'
    ) as adm
    ON bene.bene_id = adm.bene_id
    WHERE
      race in ('1', '2') AND
      sex in ('1', '2')
    ;
    """
    ## Request query ----
    %time a = pd.read_sql_query(sql_query, connection, index_col = 'zip').reset_index()
    adm_zip_list.append(a)

2000
CPU times: user 4.28 s, sys: 1.31 s, total: 5.59 s
Wall time: 4.81 s
2001
CPU times: user 6.23 s, sys: 1.5 s, total: 7.72 s
Wall time: 6.51 s
2002
CPU times: user 5.94 s, sys: 1.52 s, total: 7.46 s
Wall time: 6.22 s
2003
CPU times: user 8.01 s, sys: 1.95 s, total: 9.96 s
Wall time: 8.4 s
2004
CPU times: user 6.66 s, sys: 1.92 s, total: 8.59 s
Wall time: 7.03 s
2005
CPU times: user 8.43 s, sys: 1.98 s, total: 10.4 s
Wall time: 8.79 s
2007
CPU times: user 7.41 s, sys: 1.9 s, total: 9.31 s
Wall time: 7.68 s
2008
CPU times: user 7.68 s, sys: 1.83 s, total: 9.52 s
Wall time: 12.9 s
2009
CPU times: user 6.95 s, sys: 1.6 s, total: 8.55 s
Wall time: 14.1 s
2010
CPU times: user 8.52 s, sys: 1.91 s, total: 10.4 s
Wall time: 13.8 s
2011
CPU times: user 9.57 s, sys: 2.23 s, total: 11.8 s
Wall time: 13.8 s
2012
CPU times: user 9.94 s, sys: 2.5 s, total: 12.4 s
Wall time: 14.5 s
2013
CPU times: user 7.66 s, sys: 2.43 s, total: 10.1 s
Wall time: 11.7 s
2014
CPU times: user 11.2 s, sys: 2.67 s, t

In [10]:
adm_zip_df = pd.concat(adm_zip_list)
adm_zip_df['age'] = adm_zip_df.year - adm_zip_df.yob_
adm_zip_df['age_grp'] = pd.cut(x=adm_zip_df['age'], 
                               bins=[min(adm_zip_df.age), 65, 75, 85, max(adm_zip_df.age)],
                               labels=['<65', '[65,75)', '[75,85)', '>85'])

## read outcomes ----
read_path = '../data/input/remote/icd_codes.json'
outcomes = get_outcomes(read_path)

## find diagnoses ----
for outcome in ['adrd']:
    adm_zip_df[outcome] = [get_outcome_in_diagnoses(get_outcomes_set(outcome, y_), d_[:1]) for y_, d_ in zip(adm_zip_df.year, adm_zip_df.diagnoses)]

In [11]:
adm_zip_df[['year', 'adrd']].groupby(['year']).sum()

,adrd
year,
2000,2749
2001,2777
2002,3040
2003,4805
2004,4815
2005,4575
2007,3400
2008,3472
2009,3124


In [12]:
keep = adm_zip_df[['adrd']].any(axis=1)
adm_zip_df = adm_zip_df[keep]
adm_zip_df = adm_zip_df.drop(columns='adrd')
adm_zip_df = adm_zip_df.groupby(['year', 'zip', 'race', 'sex', 'age_grp'])['bene_id'].count().reset_index()
adm_zip_df = adm_zip_df.rename(columns = {'bene_id':'n_adrd'})

In [13]:
## crosswalk to counties ----
adm_county_df = adm_zip_df.merge(zip_to_county)
adm_county_df['n_adrd'] = adm_county_df.n_adrd * adm_county_df.w
adm_county_df = adm_county_df.groupby(['year', 'county', 'race', 'sex', 'age_grp'])['n_adrd'].sum().reset_index()

In [14]:
## total number of adrd admissions in zipcodes ----
adm_zip_df.n_adrd.sum()

55848

In [15]:
## total number of adrd admissions in counties ----
adm_county_df.n_adrd.sum()

54159.0

## Adrd counts

In [16]:
## obtain rows for all combinations of county, year, race, sex and age_grp ----
## merge with enrollee and adrd counts
## there may be missing counts for a given combination
county_ = sorted(bene_county_df.county.unique())
year_ = sorted(bene_county_df.year.unique())
race_ = sorted(bene_county_df.race.unique())
sex_ = sorted(bene_county_df.sex.unique())
age_grp_ = sorted(bene_county_df.age_grp.unique())

In [17]:
adrd_county_df = pd.DataFrame({'county':county_}).merge(pd.DataFrame({'year':year_}), how = 'cross')
adrd_county_df = adrd_county_df.merge(pd.DataFrame({'race':race_}), how = 'cross')
adrd_county_df = adrd_county_df.merge(pd.DataFrame({'sex':sex_}), how = 'cross')
adrd_county_df = adrd_county_df.merge(pd.DataFrame({'age_grp':age_grp_}), how = 'cross')

In [18]:
adrd_county_df['state'] = [str(x)[0:2] for x in adrd_county_df.county]
adrd_county_df = adrd_county_df[adrd_county_df.state == '37']

In [19]:
adrd_county_df = adrd_county_df.merge(bene_county_df, how = 'left')
adrd_county_df = adrd_county_df.merge(adm_county_df, how = 'left')

In [20]:
## total number of counties in bene_county_df (resulting from crosswalk)----
len(county_)

1125

In [21]:
## total number of counties in adrd_county_df (after filtering NC) ----
len(adrd_county_df.county.unique())

100

In [22]:
## total number of enrollees in adrd_county_df ----
adrd_county_df.n_enrollees.sum()

24311184.0

In [23]:
## total number of adrd admissions in adrd_county_df ----
adrd_county_df.n_adrd.sum()

54125.0

In [24]:
## percentage of county-race-sex-age_grp combinations with missing enrollees (all years) ----
adrd_county_df.n_enrollees.isnull().mean()

0.010073529411764705

In [25]:
## percentage of county-race-sex-age_grp combinations with missing adrd hospitalizations (all years) ----
adrd_county_df.n_adrd.isnull().mean()

0.0

In [26]:
## save adrd_county_df
adrd_county_df.to_csv("../data/input/adrd_county_df.csv", index=False)